In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tabulate import tabulate
import warnings
from sklearn_genetic import GASearchCV
from sklearn_genetic.space import Integer
from sklearn.metrics import make_scorer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split,  cross_validate
from sklearn.preprocessing import StandardScaler
import time
import sys, os
from sklearn.pipeline import make_pipeline
from skopt import BayesSearchCV
import skopt.space.space
from sklearn.model_selection import KFold


path = 'datasets/Dataset_2.xlsx'

# Read and load dataset
df= pd.read_excel(path, sheet_name="preproc")





/home/demetra/Desktop/Demetra/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INPUT DATA

In [2]:
X = (df.iloc[:,14:]).values
print(X.shape)
std_scaler = StandardScaler()

(2905, 530)


OUTPUT DATA

In [3]:
Y = (df.iloc[:,7]).values
print(Y.shape)

(2905,)


**NRMSEPiqr**

In [4]:


def NRMSEPiqr(observed_values, predicted_values):
    # Calculate RMSEP
    rmsep = np.sqrt(np.mean((observed_values - predicted_values) ** 2))
    # Calculate Q1 (25th percentile) and Q3 (75th percentile)
    Q1 = np.percentile(observed_values, 25)
    Q3 = np.percentile(observed_values, 75)

    # Calculate IQR
    IQR = Q3 - Q1

    return rmsep/IQR

NRMSEPiqrscorer = make_scorer(NRMSEPiqr,greater_is_better=False)


In [5]:
if not sys.warnoptions:
    warnings.simplefilter("ignore")
    os.environ["PYTHONWARNINGS"] = "ignore" 


trials = 100
ranndom_cv = np.random.seed(777)
seeds = np.random.choice(trials, size=trials)

ripartisions = [0.25, 0.50, 0.75]

results = []  

for ripartition in ripartisions:
        
    run_name = str(ripartition) + " ripartition"
    print(run_name)
    for i in range(trials):
        result = {"Ripartition":None,
                   "Trial":None,
                         "n_estimators_bayesian":None,
                           "n_estimators_evolutionary":None,
                            "max_depth_bayesian":None,
                            "max_depth_evolutionary":None,                           
                             "bayesian_time":None,
                             "evolutionary_time" :None,
                             "NRMSEPiqr_bayesian":None,
                              "NRMSEPiqr_evolutionary":None,
                               "seed": None}
            

        print("Trial:",i+1)
                
        result["Ripartition"] = ripartition
        result["Trial"] = i +1  
        result["seed"] = seeds[i]
        X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=ripartition, random_state=seeds[i])
        param_grid_evol = {
              'n_estimators': [100, 200, 300, 1000],
              'max_depth': Integer(100, 1000)
        }
        param_grid_bayesian = {
              'n_estimators': skopt.space.space.Integer(100,500),
              'max_depth':skopt.space.space.Integer(1,100)
        }
        rf =  RandomForestRegressor()
        
        kf = KFold(n_splits=5, shuffle=True,random_state=ranndom_cv)
        #sturges = int(1 + np.log(len(X_train)))    

        '''       
        evolved_estimator = GASearchCV(estimator=rf,
                                cv=kf,
                                scoring=NRMSEPiqrscorer,
                                param_grid=param_grid_evol,
                                n_jobs=-1,
                                verbose=False,
                                population_size=10,
                                generations=10)  
        '''
        
        bayesian_estimator = BayesSearchCV(
                                rf,
                                param_grid_bayesian,
                                cv=kf,
                                scoring=NRMSEPiqrscorer,
                                n_jobs=-1,
                                verbose= False,
                                 n_iter=15 )
            
        #start_time = time.time()
        #evolved_estimator.fit(X_train, Y_train)
        #elapsed_time = (time.time() -start_time) #elapsed time in seconds

        #result["evolutionary_time"] = elapsed_time
        #result["n_estimators_evolutionary"] = float(evolved_estimator.best_params_["n_components"])
       
        start_time = time.time()
        bayesian_estimator.fit(X_train, Y_train)
        elapsed_time = (time.time() -start_time) #elapsed time in seconds

        result["bayesian_time"] = elapsed_time
        result["n_estimators_bayesian"] = float(bayesian_estimator.best_params_["n_components"])
        result["max_depth_bayesian"] = float(bayesian_estimator.best_params_["max_depth"])
    
           
        #result["NRMSEPiqr_evolutionary"] = NRMSEPiqr(Y_test, evolved_estimator.predict(X_test))
    

        result["NRMSEPiqr_bayesian"] =  NRMSEPiqr(Y_test,bayesian_estimator.predict(X_test))
    
        print(result)
        results.append(result)

0.25 ripartition
Trial: 1


KeyboardInterrupt: 

In [ ]:

df = pd.DataFrame(results)
df.to_excel("RF_Bayes_vs_Evol.xlsx")